# Individual Exercise: Mini Data Quality Audit

Homework notebook for Week 2.


## Instructions

Perform a mini data quality audit and submit this completed notebook.


In [34]:
from pathlib import Path
import pandas as pd
import numpy as np

file_path = "week2_customer_transactions_messy.csv"
data_path = Path(file_path)

df = pd.read_csv(data_path)
print('Loaded:', data_path)
print('Shape:', df.shape)
df


Loaded: week2_customer_transactions_messy.csv
Shape: (11, 9)


,transaction_id,customer_id,transaction_date,amount,currency,payment_method,status,region,last_updated
0,T0001,C100,2026-01-05,120.50,EUR,card,completed,DE,2026-01-05
1,T0002,C101,2026/01/06,0.00,EUR,CARD,completed,de,2026-01-20
2,T0003,C102,06-01-2026,-35.00,USD,bank_transfer,completed,US,2026-01-07
3,T0004,NaN,2026-01-07,250.00,EUR,card,pending,FR,2026-01-08
4,T0005,C104,2026-01-07,89.99,EURO,cash,completed,DE,2026-01-09
5,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
6,T0006,C105,2026-01-08,19.99,EUR,card,cancelled,DE,2026-04-15
7,T0007,C106,2026-02-30,47.00,EUR,card,completed,DE,2026-02-15
8,T0008,C107,2026-01-10,1000000.00,EUR,card,completed,DE,2026-01-10
9,T0009,C108,2026-01-11,NaN,EUR,bank_transfer,completed,NL,NaN


## Required tasks

1. describe the dataset briefly
2. identify issues by dimension
3. compute at least 3 KPIs
4. define at least 3 validation rules
5. create a short audit summary
6. recommend cleaning steps for next chapter (do not implement full pipelines)


## Task 1 - Dataset description

The dataset consists of customer transactions. It contains the following fields: transaction_id, customer_id, transaction_date, amount, currency, payment_method, status, region and last_updated. Aside of that a few inconsistencies can be found just by looking at the logs. Key fields, such as the customer_id, amount or payment_method, having no value within a transaction is not acceptable. There are different values for the same fields: The currency 'EUR' also appears as 'EURO' once. The same can be observed for the region field with Germany being represented as 'DE' and 'de'.

Business Use Case:
This data can be used for revenue reporting or analysis of Customer value to the company.


## Task 2 - Issues by dimension


In [35]:
issue_table = pd.DataFrame([['Missing customer_id','Completeness','Impacts customer analytics'],
                            ['Duplicate transaction_id','Uniqueness','May double count revenue'],
                            ['Negative / Missing Amounts', 'Validity', 'Negative amounts might be entry errors or valid values.'],
                            ['Invalid Dates ("2026-02-30")', 'Validity', 'Unparseable dates will disrupt time-series analyses or monthly aggregations.'],
                            ['Timeliness last_updated', 'Timeliness', 'The time between transaction and the last update of one is over seven days indicating a delay in updating the data.'],
                            ['Inconsistent Values / Formats (DE and de)', 'Consistency', 'Different text casing or non-standard abbreviations (e.g. EURO vs EUR) group same categories separately']], 
                            columns=['Issue','Dimension','Impact'])
issue_table


,Issue,Dimension,Impact
0,Missing customer_id,Completeness,Impacts customer analytics
1,Duplicate transaction_id,Uniqueness,May double count revenue
2,Negative / Missing Amounts,Validity,Negative amounts might be entry errors or vali...
3,"Invalid Dates (""2026-02-30"")",Validity,Unparseable dates will disrupt time-series ana...
4,Timeliness last_updated,Timeliness,The time between transaction and the last upda...
5,Inconsistent Values / Formats (DE and de),Consistency,Different text casing or non-standard abbrevia...


## Task 3 - KPI calculations


In [36]:
kpis={}
kpis['Completeness Rate']=1-(df.isna().sum().sum()/(df.shape[0]*df.shape[1]))

kpis['Duplication Rate']=df.duplicated(subset=['transaction_id']).mean()

amount=pd.to_numeric(df['amount'], errors='coerce')
trx = pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed')
upd = pd.to_datetime(df['last_updated'], errors='coerce', format='mixed')

kpis['Error Rate'] = (amount.isna() | (amount < 0) | trx.isna()).mean()

valid_dates = trx.notna() & upd.notna()
lag = (upd - trx).dt.days
timely = valid_dates & (lag >= 0) & (lag <= 7)

kpis['Timeliness Rate'] = timely.sum() / valid_dates.sum() if valid_dates.sum() > 0 else 0

kpis['Overall Quality Score'] = (
    0.30 * kpis['Completeness Rate'] + 
    0.30 * (1 - kpis['Error Rate']) + 
    0.20 * (1 - kpis['Duplication Rate']) + 
    0.20 * kpis['Timeliness Rate']
)

kpi_df = pd.DataFrame({'KPI': list(kpis.keys()), 'Value': list(kpis.values())})
kpi_df


,KPI,Value
0,Completeness Rate,0.959596
1,Duplication Rate,0.090909
2,Error Rate,0.272727
3,Timeliness Rate,0.555556
4,Overall Quality Score,0.798990


### KPI interpretation
The data quality audit indicates an Overall Quality Score of 79.90%, suggesting that the dataset is only partially reliable and should be used with caution. Although the score shows that the data is usable to some extent, there are notable risks that could affect the accuracy of financial reporting and the validity of time-based analyses.

The Completeness Rate stands at 95.96%, which means that the overall amount of missing data is relatively low. However, the missing values are concentrated in particularly important fields such as customer_id and amount. Records missing either the transaction amount or the associated customer cannot be meaningfully used.

The Duplication Rate is 9.09%, meaning that approximately one out of every eleven records contains a duplicated transaction_id. This represents a serious issue in terms of data uniqueness. If these duplicates remain unresolved, they would artificially inflate reported sales volume and revenue, leading to misleading business insights.

The Error Rate of 27.27% is the most concerning finding in the audit. More than one quarter of all records contain a critical error, such as a negative or missing transaction amount, or an invalid date such as February 30th. This strongly suggests insufficient front-end validation controls in the source system where the data was originally entered.

Finally, the Timeliness Rate is only 55.56%, indicating that just over half of the records were updated within the intended seven-day period following the transaction date. The considerable delay between transaction_date and last_updated points to possible batch-processing delays or manual reconciliation procedures, both of which could limit the organisation’s ability to generate timely, real-time business intelligence.



## Task 4 - Validation rules


In [37]:
rules={
'transaction_id_required': df['transaction_id'].isna() | (df['transaction_id'].astype(str).str.strip()==''),
'amount_non_negative': (pd.to_numeric(df['amount'], errors='coerce') < 0) | (df['amount'].isna()),
'transaction_date_valid': pd.to_datetime(df['transaction_date'], errors='coerce', format='mixed').isna(),
'customer_id_missing': df['customer_id'].isna()
}
pd.DataFrame({k:int(v.sum()) for k,v in rules.items()}, index=['affected_rows']).T


,affected_rows
transaction_id_required,0
amount_non_negative,2
transaction_date_valid,1
customer_id_missing,1


## Task 5 - Audit summary


In [38]:
audit_summary = pd.DataFrame([
    ['Duplicate transaction_id', 1, 'High', 'Drop the duplicate row, keeping the first occurrence.'],
    ['Negative/Missing Amount', 2, 'High', 'Investigate negative values as potential refunds. Drop or impute missing amounts.'],
    ['Invalid Transaction Date', 1, 'High', 'Coerce dates to standard formats. Drop dates that do not exist (e.g., Feb 30th).'],
    ['Missing customer_id', 1, 'Medium', 'Assign to a default "Guest" ID if the revenue is valid, or flag for investigation.']
], columns=['issue_type', 'affected_rows', 'severity', 'recommended_next_action'])

audit_summary

,issue_type,affected_rows,severity,recommended_next_action
0,Duplicate transaction_id,1,High,"Drop the duplicate row, keeping the first occu..."
1,Negative/Missing Amount,2,High,Investigate negative values as potential refun...
2,Invalid Transaction Date,1,High,Coerce dates to standard formats. Drop dates t...
3,Missing customer_id,1,Medium,"Assign to a default ""Guest"" ID if the revenue ..."


## Task 6 - Recommended cleaning steps for next chapter

- Deduplication of rows with duplicate transaction_ids. IDs are unique identifiers therefore they should be unique.
- Date Standardisation and Validation
- Financial Value Correction (amount shouldn't be negative except if the company pays customers back)
- Field Data Harmonisation (no different values for the same category such as e.g. 'EUR' and 'EURO' in currency)
- Missing Value Strategy (Imputation/Removal)
- Regional Standardisation (Region column uses a consistent standard or format.)


## Reflection questions

1. Which KPI gave the strongest signal?
The Error Rate (27.27%) gave the strongest signal. This indicates that over a quarter of the data contains logical errors.

2. Which issue should be escalated first?
The duplicate transaction_id should be escalated first, as too many rows, due to a non-unique key, can lead to errors in financial reporting.

3. What additional metadata would improve this audit?
A Data Dictionary or schema would help to explain why the amount was 0 or negative when a transaction has the status 'cancelled'. Another solution would be to have Refund or Return Flags which would also explain the negative amount values.

## Bonus (recommended): write your own audit function

Create at least one helper function with clear parameters and docstring.
This demonstrates that your logic is reusable and understandable.


In [40]:
def summarize_rule_violations(rule_dictionary, sort_ascending=False):
    """Summarize affected-row counts for each validation rule.

    Parameters
    ----------
    rule_dictionary : dict[str, pd.Series]
        Dictionary where keys are rule names and values are boolean masks (True = violation).
    sort_ascending : bool, default False
        Whether to sort the results from lowest to highest violation count.

    Returns
    -------
    pd.DataFrame
        A summary table with rule names and the total count of rows violating each rule.
    """
    summary = pd.DataFrame({
        'rule_name': list(rule_dictionary.keys()),
        'affected_rows': [int(mask.sum()) for mask in rule_dictionary.values()]
    })
    return summary.sort_values('affected_rows', ascending=sort_ascending)

# Example usage:
summarize_rule_violations(rules, sort_ascending=False)

,rule_name,affected_rows
1,amount_non_negative,2
2,transaction_date_valid,1
3,customer_id_missing,1
0,transaction_id_required,0


### Explain your function parameters

In 3-5 lines, explain:
- what each parameter means,
- why you selected default values,
- how changing parameters affects KPI/audit results.


### Explanation
The rule_dictionary parameter contains a mapping of descriptive rule names to the boolean validation masks created during the checking process. This modular structure makes it easy to add new rules, such as invalid_currency, without rewriting the reporting logic. The sort_ascending parameter is set to False by default so that the most serious issues, affecting the highest number of rows, appear first in the results. If changed to True, an auditor could instead quickly identify rules with zero violations. Updating the dictionary automatically refreshes the KPI results, making the function reusable for future transaction datasets.

## Submission checklist

- [ x ] Dataset described
- [ x ] Issues mapped to dimensions
- [ x ] At least 3 KPIs calculated
- [ x ] At least 3 validation rules defined
- [ x ] Audit summary completed
- [ x ] Cleaning recommendations proposed (not implemented)
